# Probabilistic & Approximate Structures

Every structure so far has been **exact**: a `set` knows precisely which keys it holds, a `Counter` knows exactly how many times it saw each item. That exactness costs **O(n) space** — one stored entry per distinct key. Probabilistic structures make a different bargain: they accept a small, **bounded, tunable error** in exchange for space and time that can be *orders of magnitude* smaller. This is how real systems answer "have I seen this URL?", "how many unique visitors?", and "which keys are hot?" at web scale, where storing every key is simply off the table. We build three from scratch and **measure** that the error matches the theory.

**Contents**

1. **What & why** — trading exactness for space
2. **Bloom filter** — set membership with no false negatives
3. **Skip list** — a randomized balanced-tree alternative
4. **Count-min sketch** — frequency estimation in sublinear space
5. **HyperLogLog** — counting distinct items in kilobytes
6. **When to use what**
7. **Cheat-sheet** — space / time / error

## 1. What & why — trading exactness for space

An exact structure must, in the worst case, **store something for every distinct key** — that's an information-theoretic floor, not an implementation detail. If you want to answer membership/count/cardinality questions over a billion keys with perfect accuracy, you pay for a billion keys' worth of memory.

Approximate structures escape that floor by being allowed to be **wrong a little**, in a controlled direction:

| Structure | Answers | Error shape | Exact cousin |
|---|---|---|---|
| **Bloom filter** | is `x` in the set? | false *positives* only (never false negatives) | `set` |
| **Count-min sketch** | how many times did `x` appear? | *over*estimates only (never under) | `collections.Counter` |
| **HyperLogLog** | how many *distinct* items? | bounded relative error (±a few %) | `len(set(...))` |

The skip list is the odd one out — it's **exact**, but uses *randomization* (coin flips, not hashing) to get balanced-tree performance without the bookkeeping of an AVL/red-black tree. It belongs here because it's the canonical *probabilistic* data structure and powers Redis sorted sets.

The recurring trick in the approximate three is **hashing** — the same machinery behind `dict` and `set` (see the **dict & set** notebook's hashing discussion). We lean on a handful of hash functions to scatter keys across a small fixed array; collisions are what create the error, and the math tells us exactly how much.

## 2. Bloom filter — membership with no false negatives

A **Bloom filter** answers *"have I seen `x`?"* using a bit array of `m` bits and `k` hash functions. It cannot tell you *what* it holds, only whether something is **probably present** or **definitely absent**:

- **add(x):** set the `k` bits at positions `h_1(x) … h_k(x)` to 1.
- **query(x):** if *any* of those `k` bits is 0, `x` is **definitely not** present (no false negatives — you'd have set them on insert). If *all* are 1, `x` is **probably** present — but those bits might have been set by other items, a **false positive**.

**No false negatives, tunable false positives** — that asymmetry is the whole point. A CDN can keep a Bloom filter of cached URLs in a few kilobytes: a "no" is trustworthy (skip the cache), a "yes" just means "go check".

### Double hashing

We don't need `k` independent hash functions. The standard trick (Kirsch–Mitzenmacher) derives all `k` indices from **two** base hashes: `g_i(x) = (h1(x) + i·h2(x)) mod m`. We get `h1` and `h2` by slicing one SHA-256 digest in half — cheap, and statistically good enough.

In [1]:
import hashlib, math, random

class BloomFilter:
    """Bit array backed by a bytearray; k indices via double hashing."""
    def __init__(self, m_bits, k):
        self.m = m_bits
        self.k = k
        self.bits = bytearray((m_bits + 7) // 8)   # m bits, packed 8 per byte
        self.n = 0                                  # items inserted

    def _indices(self, item):
        b = item.encode() if isinstance(item, str) else item
        digest = hashlib.sha256(b).digest()
        h1 = int.from_bytes(digest[:8], "big")      # two base hashes from
        h2 = int.from_bytes(digest[8:16], "big")    # one digest (double hashing)
        for i in range(self.k):
            yield (h1 + i * h2) % self.m

    def add(self, item):
        for pos in self._indices(item):
            self.bits[pos >> 3] |= (1 << (pos & 7))  # set bit `pos`
        self.n += 1

    def __contains__(self, item):
        return all(self.bits[pos >> 3] & (1 << (pos & 7))
                   for pos in self._indices(item))

bf = BloomFilter(m_bits=100, k=3)
bf.add("apple"); bf.add("banana")
print("apple  in bf :", "apple"  in bf)     # True  (inserted)
print("cherry in bf :", "cherry" in bf)     # almost certainly False
print("bits set     :", sum(bin(b).count("1") for b in bf.bits), "/ 100")

apple  in bf : True
cherry in bf : False
bits set     : 6 / 100


### Sizing: the optimal `m` and `k`

Given an expected item count `n` and a target false-positive rate `p`, the optimal bit-array size and hash count are:

$$m = -\frac{n\ln p}{(\ln 2)^2}, \qquad k = \frac{m}{n}\ln 2$$

Two takeaways fall out of these formulas. First, the cost is **~1.44·log₂(1/p) bits per item** — to get `p = 1%` you spend under 10 bits per key, *regardless of how big the keys are*. Second, the optimal `k` is independent of `p` at fixed `m/n`; more hashes isn't always better, because each one also sets more bits and fills the array faster.

In [2]:
def optimal_params(n, p):
    m = math.ceil(-n * math.log(p) / (math.log(2) ** 2))
    k = max(1, round((m / n) * math.log(2)))
    return m, k

for p in (0.1, 0.01, 0.001):
    m, k = optimal_params(10_000, p)
    print(f"p={p:<6}  m={m:6d} bits  ({m/10_000:5.2f} bits/item)  k={k}")

p=0.1     m= 47926 bits  ( 4.79 bits/item)  k=3
p=0.01    m= 95851 bits  ( 9.59 bits/item)  k=7
p=0.001   m=143776 bits  (14.38 bits/item)  k=10


### The fact-check: measured false-positive rate vs theory

Now the claim that matters. Insert `n` items into an optimally-sized filter, then query a large batch of **held-out** items (guaranteed never inserted) and count how many wrongly come back positive. The theoretical rate after `n` insertions is:

$$p_{\text{fp}} = \left(1 - e^{-kn/m}\right)^{k}$$

We also assert **zero false negatives** — that's a hard guarantee, not a probability. RNG is seeded for reproducibility.

In [3]:
random.seed(42)

n, target_p = 5_000, 0.01
m, k = optimal_params(n, target_p)
bf = BloomFilter(m, k)

inserted = [f"item-{i}" for i in range(n)]
for it in inserted:
    bf.add(it)

# (1) HARD GUARANTEE: every inserted item must be found -> no false negatives.
assert all(it in bf for it in inserted), "Bloom filter produced a false negative!"

# (2) MEASURE false positives on 100k held-out items that were never inserted.
trials = 100_000
false_pos = sum(1 for i in range(trials) if f"held-out-{i}" in bf)
measured = false_pos / trials
theory   = (1 - math.exp(-k * n / m)) ** k

print(f"m={m} bits, k={k}, n={n} inserted")
print("false negatives        : 0  (guaranteed)")
print(f"measured false-positive: {measured:.4f}")
print(f"theoretical  (formula) : {theory:.4f}")
print(f"design target          : {target_p}")
assert abs(measured - theory) < 0.003, "measurement drifted from theory!"
print("measured matches theory within 0.3% -> claim verified")

m=47926 bits, k=7, n=5000 inserted
false negatives        : 0  (guaranteed)
measured false-positive: 0.0104
theoretical  (formula) : 0.0100
design target          : 0.01
measured matches theory within 0.3% -> claim verified


The measured rate lands right on the theoretical curve, and both sit at the design target of 1%. Some practical gotchas:

- **You can't delete** from a plain Bloom filter — clearing a bit might unset a bit another item relies on, breaking the no-false-negative guarantee. (A *counting* Bloom filter replaces bits with small counters to allow removal, at extra space.)
- **Filling past `n` degrades it gracefully** — every extra insert nudges the false-positive rate up; it never suddenly breaks, but it drifts off the target you sized for.
- **It can't enumerate its members** or give you the items back — it only answers yes/no. For that you'd keep a real `set`.

## 3. Skip list — a randomized balanced-tree alternative

A sorted **linked list** has O(n) search — you walk it node by node. A **skip list** fixes that by stacking *express lanes* on top: each node gets a randomized **tower height**, and higher levels link across more nodes, so a search drops down through the levels and skips huge swaths in one hop. With tower heights drawn from a geometric distribution (`p = 0.5`: flip a coin, keep going up while it's heads), search, insert, and delete are all **O(log n) expected**.

This is the **probabilistic answer to the tree-balancing problem** (see the **trees** notebook). A red-black or AVL tree guarantees balance through intricate rotation logic; a skip list just *flips coins* and lets probability keep it balanced on average — far simpler to implement correctly. It's what **Redis uses for sorted sets** (`ZSET`), and it's easy to make concurrent.

In [4]:
class SkipNode:
    __slots__ = ("key", "forward")
    def __init__(self, key, level):
        self.key = key
        self.forward = [None] * (level + 1)    # forward[i] = next node at level i

class SkipList:
    def __init__(self, max_level=16, p=0.5, seed=0):
        self.max_level = max_level
        self.p = p
        self.head = SkipNode(None, max_level)  # sentinel, full height
        self.level = 0                         # highest level currently in use
        self._rng = random.Random(seed)

    def _random_level(self):
        lvl = 0
        while self._rng.random() < self.p and lvl < self.max_level:
            lvl += 1                            # geometric: ~half stop each step
        return lvl

    def insert(self, key):
        update = [self.head] * (self.max_level + 1)   # node before key at each level
        x = self.head
        for i in range(self.level, -1, -1):           # descend, recording turn-offs
            while x.forward[i] and x.forward[i].key < key:
                x = x.forward[i]
            update[i] = x
        if x.forward[0] and x.forward[0].key == key:
            return                                    # no duplicates
        lvl = self._random_level()
        if lvl > self.level:
            self.level = lvl
        node = SkipNode(key, lvl)
        for i in range(lvl + 1):                      # splice in at each of its levels
            node.forward[i] = update[i].forward[i]
            update[i].forward[i] = node

    def __contains__(self, key):
        x = self.head
        for i in range(self.level, -1, -1):
            while x.forward[i] and x.forward[i].key < key:
                x = x.forward[i]
        x = x.forward[0]
        return x is not None and x.key == key

    def delete(self, key):
        update = [self.head] * (self.max_level + 1)
        x = self.head
        for i in range(self.level, -1, -1):
            while x.forward[i] and x.forward[i].key < key:
                x = x.forward[i]
            update[i] = x
        x = x.forward[0]
        if not x or x.key != key:
            return False
        for i in range(self.level + 1):
            if update[i].forward[i] is x:
                update[i].forward[i] = x.forward[i]
        while self.level > 0 and self.head.forward[self.level] is None:
            self.level -= 1                           # shrink unused top levels
        return True

    def __iter__(self):                               # level-0 chain == sorted order
        x = self.head.forward[0]
        while x:
            yield x.key
            x = x.forward[0]

    def search_path(self, key):
        """Return the (level, node-key) turn-off points a search visits."""
        path, x = [], self.head
        for i in range(self.level, -1, -1):
            while x.forward[i] and x.forward[i].key < key:
                x = x.forward[i]
            path.append((i, x.key))
        return path

print("SkipList defined")

SkipList defined


### Build one and verify ordering against `sorted`

The level-0 chain is just a sorted linked list, so iterating the skip list must reproduce `sorted()` exactly — that's our correctness oracle. We also round-trip membership and a delete.

In [5]:
random.seed(1)
sl = SkipList(seed=1)
data = random.sample(range(1000), 200)
for k in data:
    sl.insert(k)

assert list(sl) == sorted(data), "skip list iteration is not sorted!"
assert all(k in sl for k in data), "missing an inserted key!"
assert 1001 not in sl
print("iteration == sorted(data):", list(sl)[:10], "...")

removed = data[0]
assert sl.delete(removed) and removed not in sl
assert list(sl) == sorted(k for k in data if k != removed)
print(f"deleted {removed}; still sorted, {len(list(sl))} keys remain")

iteration == sorted(data): [1, 2, 5, 9, 12, 22, 26, 28, 29, 30] ...
deleted 137; still sorted, 199 keys remain


### Watching a search skip

Searching for a key descends from the top level, racing forward on each express lane until the next node would overshoot, then dropping a level. The turn-off points show the search **skipping past** most of the list instead of stepping through it one node at a time.

In [6]:
sl2 = SkipList(seed=2)
for k in range(0, 200, 2):      # even numbers 0..198
    sl2.insert(k)

print("search path for 137 (level: key we stopped at before dropping down):")
for lvl, key in sl2.search_path(137):
    print(f"  level {lvl}: advanced to {key}")
print("137 in list:", 137 in sl2, " 136 in list:", 136 in sl2)

search path for 137 (level: key we stopped at before dropping down):
  level 7: advanced to 24
  level 6: advanced to 24
  level 5: advanced to 24
  level 4: advanced to 134
  level 3: advanced to 134
  level 2: advanced to 134
  level 1: advanced to 134
  level 0: advanced to 136
137 in list: False  136 in list: True


### Why it stays balanced: the geometric height distribution

Balance isn't enforced — it *emerges* from the coin flips. With `p = 0.5`, about half the nodes have a tower of height 1, a quarter height 2, an eighth height 3, and so on. That geometric decay means the top levels hold only a handful of nodes (the coarse express lanes) while the bottom holds everyone, giving the O(log n) layering for free.

In [7]:
from collections import Counter

random.seed(0)
big = SkipList(max_level=20, seed=0)
for k in range(10_000):
    big.insert(k)

heights = Counter()
x = big.head.forward[0]
while x:
    heights[len(x.forward)] += 1     # tower height of each node
    x = x.forward[0]

print("height : count  (each level should be ~half the one below)")
prev = None
for h in sorted(heights):
    ratio = f"{prev / heights[h]:.2f}x" if prev else "  -"
    print(f"  {h:2d}   : {heights[h]:5d}   (~{ratio} the level below)")
    prev = heights[h]
print(f"\nhighest level in use: {big.level}  vs  log2(10000) = {math.log2(10000):.1f}")

height : count  (each level should be ~half the one below)
   1   :  5047   (~  - the level below)
   2   :  2468   (~2.04x the level below)
   3   :  1243   (~1.99x the level below)
   4   :   642   (~1.94x the level below)
   5   :   321   (~2.00x the level below)
   6   :   131   (~2.45x the level below)
   7   :    80   (~1.64x the level below)
   8   :    29   (~2.76x the level below)
   9   :    18   (~1.61x the level below)
  10   :    12   (~1.50x the level below)
  11   :     6   (~2.00x the level below)
  12   :     2   (~3.00x the level below)
  13   :     1   (~2.00x the level below)

highest level in use: 12  vs  log2(10000) = 13.3


Each level holds roughly half the nodes of the one below — exactly the geometric decay the coin flips produce — and the tallest tower tracks log₂(n). A practical note: in pure Python a skip list's O(log n) **pointer-chasing happens in the interpreter**, so for raw wall-clock insert speed `bisect.insort` into a `list` (an O(n) but C-level `memmove`) often *beats* it at modest sizes. The skip list's real edges are **asymptotics at scale** and **O(log n) inserts/deletes in the middle** (no array shift), which is why C implementations like Redis favor it. For application code, reach for `sortedcontainers` (covered in the **trees** notebook) before hand-rolling one.

## 4. Count-min sketch — frequency estimation in sublinear space

A `Counter` stores one entry per distinct key. Over a stream with millions of distinct keys — IP addresses, search terms, product IDs — that's a lot of memory, most of it spent on **cold keys you don't care about**. The **count-min sketch (CMS)** estimates every key's frequency in a **fixed-size** table that doesn't grow with the number of distinct keys.

It's a `d × w` grid of counters with `d` independent hash functions (one per row):

- **add(x):** in each row `r`, increment the cell `table[r][h_r(x)]`.
- **estimate(x):** return the **minimum** over the `d` rows of `table[r][h_r(x)]`.

Why the min? Each cell may have been bumped by *other* keys that collided there, so every row gives an **over**estimate. Collisions only ever *add* count, never remove it — so the true value is a lower bound, and the smallest (least-collided) row is the tightest estimate. Hence: **overestimates only, never underestimates.**

In [8]:
class CountMinSketch:
    def __init__(self, w, d, seed=0):
        self.w, self.d = w, d
        self.table = [[0] * w for _ in range(d)]
        rnd = random.Random(seed)
        self.seeds = [rnd.randrange(1 << 30) for _ in range(d)]   # one per row

    def _col(self, key, r):
        b = f"{key}:{self.seeds[r]}".encode()
        h = hashlib.blake2b(b, digest_size=8).digest()
        return int.from_bytes(h, "big") % self.w

    def add(self, key, count=1):
        for r in range(self.d):
            self.table[r][self._col(key, r)] += count

    def estimate(self, key):
        return min(self.table[r][self._col(key, r)] for r in range(self.d))

cms = CountMinSketch(w=50, d=4, seed=1)
for word in "the cat sat on the mat the cat ran".split():
    cms.add(word)
print("the:", cms.estimate("the"), " cat:", cms.estimate("cat"),
      " dog:", cms.estimate("dog"))

the: 3  cat: 2  dog: 0


### The error bound, and the fact-check

Pick the table shape from two knobs: an error fraction `ε` and a failure probability `δ`, via `w = ⌈e/ε⌉` and `d = ⌈ln(1/δ)⌉`. The guarantee: with probability ≥ `1 − δ`, the estimate overshoots the true count by at most **`ε·N`**, where `N` is the total number of items in the stream.

We stream a **skewed** dataset — a few heavy hitters plus a long tail of 40k cold singletons (which force the collisions that create error) — and check three claims directly: (1) never underestimates, (2) every estimate is within `ε·N` of truth, (3) heavy hitters come back essentially exact while cold keys get a small collision-driven bump.

In [9]:
from collections import Counter

random.seed(7)
heavy = {"hot-0": 50_000, "hot-1": 30_000, "hot-2": 12_000,
         "hot-3": 5_000, "hot-4": 800}
stream = []
for key, c in heavy.items():
    stream += [key] * c
for i in range(40_000):                       # long tail of cold keys
    stream += [f"cold-{i}"] * random.randint(1, 3)
random.shuffle(stream)

true = Counter(stream)
N = len(stream)
print(f"stream: {N:,} items, {len(true):,} distinct keys")

w, d = 2_000, 5
eps = math.e / w                              # implied error fraction
bound = eps * N
cms = CountMinSketch(w, d, seed=123)
for x in stream:
    cms.add(x)

# (1) NEVER underestimates.
assert all(cms.estimate(k) >= true[k] for k in true), "underestimate -- impossible!"
# (2) Every estimate within eps*N of truth.
max_over = max(cms.estimate(k) - true[k] for k in true)
assert max_over <= bound, "exceeded the eps*N bound!"

print(f"table {w}x{d} = {w*d:,} counters   (vs {len(true):,} Counter entries)")
print(f"error bound eps*N = {bound:.0f}   max overestimate observed = {max_over}")
print()
print("HEAVY hitters -- near exact:")
for k in heavy:
    print(f"  {k:6s}  true={true[k]:6d}  est={cms.estimate(k):6d}  over=+{cms.estimate(k)-true[k]}")

cold_over = [cms.estimate(f"cold-{i}") - true[f"cold-{i}"] for i in range(40_000)]
print(f"\nCOLD keys -- collision bump: mean over=+{sum(cold_over)/len(cold_over):.1f}, "
      f"max over=+{max(cold_over)} (all <= bound {bound:.0f})")

stream: 177,764 items, 40,005 distinct keys


table 2000x5 = 10,000 counters   (vs 40,005 Counter entries)
error bound eps*N = 242   max overestimate observed = 55

HEAVY hitters -- near exact:
  hot-0   true= 50000  est= 50043  over=+43
  hot-1   true= 30000  est= 30028  over=+28
  hot-2   true= 12000  est= 12035  over=+35
  hot-3   true=  5000  est=  5042  over=+42
  hot-4   true=   800  est=   819  over=+19

COLD keys -- collision bump: mean over=+29.1, max over=+55 (all <= bound 242)


Exactly as advertised: **no underestimate ever**, heavy hitters return with tiny relative error, cold keys pick up a small additive bump from collisions, and **every** estimate stays inside `ε·N`. The space win is the headline — `w·d` fixed counters regardless of how many distinct keys stream by. The catch: error is *additive* in `N`, so CMS shines for **finding the heavy hitters** (where the `ε·N` slop is small relative to a big true count) and is poor at pinning down rare keys (where the bump can dwarf the real value). For exact counts over a manageable key set, a `Counter` is still the right tool.

## 5. HyperLogLog — counting distinct items in kilobytes

How many **distinct** users visited today? Exactly answering that needs a `set` of every user ID — O(n) memory. **HyperLogLog (HLL)** estimates cardinality in a *fixed*, tiny footprint (a few KB handles billions of items, ~2% error) using one beautiful observation about randomness:

> In a stream of uniformly random hashes, seeing a value whose binary form > starts with `ρ` leading zeros is evidence you've drawn roughly `2^ρ` > distinct values — a run of `ρ` zeros shows up about once every `2^ρ` > draws.

So the **maximum** number of leading zeros seen is a (noisy) log₂ of the cardinality. HLL tames the noise by splitting the hash: the first `p` bits pick one of `m = 2^p` **buckets**, and each bucket tracks the max leading-zero rank of the hashes that landed in it. Averaging across buckets (via a bias-corrected harmonic mean) collapses the variance, giving a standard error of about **1.04 / √m**.

In [10]:
def hyperloglog(items, p=12):
    m = 1 << p                                  # number of buckets
    registers = [0] * m
    for it in items:
        h = int.from_bytes(hashlib.sha1(str(it).encode()).digest()[:8], "big")
        bucket = h & (m - 1)                    # low p bits -> bucket
        rest = h >> p                           # remaining bits
        rank = (64 - p) - rest.bit_length() + 1 # position of leftmost 1 (=leading zeros+1)
        registers[bucket] = max(registers[bucket], rank)
    alpha = 0.7213 / (1 + 1.079 / m)            # bias correction
    harmonic = sum(2.0 ** -r for r in registers)
    est = alpha * m * m / harmonic
    empty = registers.count(0)                  # small-cardinality correction
    if est <= 2.5 * m and empty:
        est = m * math.log(m / empty)
    return est

random.seed(99)
for true_card in (1_000, 50_000, 200_000):
    items = {f"user-{random.randrange(10**9)}" for _ in range(true_card)}
    est = hyperloglog(items, p=12)
    err = abs(est - len(items)) / len(items) * 100
    print(f"true={len(items):7,}  HLL est={est:9,.0f}  error={err:4.2f}%  "
          f"(theoretical SE = {1.04/math.sqrt(1<<12)*100:.2f}%)")

true=  1,000  HLL est=      991  error=0.93%  (theoretical SE = 1.62%)
true= 49,999  HLL est=   50,829  error=1.66%  (theoretical SE = 1.62%)


true=199,984  HLL est=  204,748  error=2.38%  (theoretical SE = 1.62%)


With `p = 12` we use 4096 one-byte registers — about **4 KB** — yet estimate hundreds of thousands of distinct items to within ~1–2%, matching the `1.04/√m` standard error. Two properties make HLL a workhorse for analytics: registers only ever take a `max`, so HLLs are trivially **mergeable** (union two days' counts by taking the elementwise max — no re-scan), and the footprint is **constant** no matter the true cardinality. Redis exposes exactly this as `PFADD` / `PFCOUNT`.

## 6. When to use what

| You need... | Reach for | Instead of | Why |
|---|---|---|---|
| Membership where a rare false "yes" is OK | **Bloom filter** | `set` | bits/item, no false negatives; can't enumerate |
| Membership *with deletes* | counting Bloom filter | `set` | counters instead of bits, more space |
| Exact membership, must list members | `set` / `dict` | — | Bloom can't give items back |
| Approximate top-K / heavy hitters | **count-min sketch** | `Counter` | fixed space; overestimates only |
| Exact counts, manageable key set | `collections.Counter` | — | CMS error is additive in N |
| Distinct count over a huge stream | **HyperLogLog** | `len(set(...))` | KBs for billions; mergeable |
| Sorted, ordered, range queries | **skip list** / `sortedcontainers` | balanced BST | randomized balance, simple; what Redis ZSET uses |
| A sorted container in app code | `sortedcontainers.SortedList` | hand-rolled skip list | battle-tested C-fast; see the **trees** notebook |

## 7. Cheat-sheet — space / time / error

<sub>n = items inserted, m = bit-array size, k = hashes, w·d = sketch cells, N = stream length, M = number of registers (=2^p).</sub>

| Structure | Space | add | query | Error |
|---|:---:|:---:|:---:|---|
| **Bloom filter** | m bits (~1.44·log₂(1/p)·n) | O(k) | O(k) | FP `(1−e^{−kn/m})^k`; **no false negatives** |
| **Skip list** | O(n) (≈2 ptrs/node avg) | O(log n) exp | O(log n) exp | **exact** (randomized balance) |
| **Count-min sketch** | w·d counters | O(d) | O(d) | ≤ `εN` over, prob ≥ `1−δ`; **never under** |
| **HyperLogLog** | M registers (~M bytes) | O(1) | O(M) | ±`1.04/√M` relative |
| `set` (for contrast) | O(n) entries | O(1) avg | O(1) avg | exact |
| `Counter` (for contrast) | O(distinct) entries | O(1) avg | O(1) avg | exact |

**The one-line mental model:** exact structures pay O(n) space for zero error; probabilistic structures fix the space (or simplify the balancing) and let a *bounded, one-directional* error absorb the difference — Bloom never says "no" wrongly, count-min never undercounts, HyperLogLog stays within a few percent. Know which direction the error leans, size for your target, and **measure it** — as every demo above did.